# 04 — Interpretation: what drives predicted risk?

**RQ2:** Which environmental features contribute most strongly to predicted H5N1 detection risk?

Approach: permutation importance on the full-data Random Forest (drop a feature, measure out-of-fold AUC loss, repeated 30×), then a predicted-risk surface for communication. (Optional upgrade: SHAP — listed in requirements.txt.)

In [1]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

import numpy as np
import pandas as pd
from sklearn.inspection import permutation_importance
from src import config
from src.models import fit_full
from src import viz

from src.features import load_panel
panel = load_panel()

## 1. Fit full model and compute permutation importance

In [2]:
rf, d = fit_full(panel, config.FULL_FEATURES, target=config.TARGET)
perm = permutation_importance(rf, d[config.FULL_FEATURES], d[config.TARGET],
                              scoring="roc_auc", n_repeats=30, random_state=42, n_jobs=-1)
viz.importance_plot(perm, config.FULL_FEATURES, config.FIGURES/"fig5_feature_importance.png")

## 2. Predicted county-level risk surface (mean monthly probability)

In [3]:
d = d.copy()
d["risk"] = rf.predict_proba(d[config.FULL_FEATURES])[:, 1]
risk = d.groupby("county").agg(lat=("lat","first"), lon=("lon","first"), risk=("risk","mean")).reset_index()
viz.risk_map(risk, config.FIGURES/"fig6_predicted_risk_map.png")

## 3. Limitations (stated plainly — this is what makes the work credible)

1. **Ecological design** — county-month associations, not individual-level causation.
2. **Surveillance bias** — APHIS detections scale with sampling intensity; ideally add an effort covariate (e.g., number of birds tested per county-month).
3. **Proxy measurement** — FIRMS sees active fire, not smoke plumes or severity; NDVI/LST are coarse habitat proxies.
4. **Residual spatial dependence** — county-grouped CV reduces leakage but not unmeasured confounding.
5. **Temporal alignment** — 1-month lag is pragmatic; mechanistic alignment needs telemetry (e.g., McDuie et al. 2024).

## 4. Conclusion

This project demonstrates an end-to-end, reproducible workflow — ecological question → multi-source spatial data integration (satellite + climate + surveillance) → baseline-vs-ML modeling under spatially explicit validation → uncertainty quantification → interpreted risk communication — that maps directly onto the EpiPandit Lab's climate-informed disease risk modeling agenda and its WildAlert / AtlasArena platforms.

**Next steps with real data:** swap in APHIS + FIRMS + MODIS + PRISM + CDL (`data/README.md`), then extend to grid-cell resolution, add waterfowl occurrence (eBird/telemetry) as the mechanistic intermediate layer, and package the risk surface as a decision-support dashboard (Streamlit).